## Temporary Assignment Work

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../data/creditsense_applications.csv")

In [ ]:
df.shape

In [ ]:
df.info()

### Observations:

- We have two columns with non-null values.
- 27 columns including one which will be ignored thanks to it being an identifier (application_id)

In [ ]:
df.isna().sum()

Same as above observation.

In [ ]:
(df.isna().mean() * 100).round(2)

Checked percentages of non-null values to assess the implications of them.

In [ ]:
df[
    df['years_at_current_address'].isna()
    & df['savings_balance_gbp'].isna()
].shape[0]

Identified the rows with both missing values which drastically changed the way I viewed the non-nulls as not many match!

In [ ]:
df.groupby(
    df['savings_balance_gbp'].isna()
)['default_within_24m'].agg(['count', 'mean'])

In [ ]:
df.groupby(
    df['years_at_current_address'].isna()
)['default_within_24m'].agg(['count', 'mean'])

Compared default rates for missing versus recorded values on savings and years at current address to asses the implication of missing values in more detail.

### Thoughts:

- Association is not the same as causation.
- Further investigation will be needed to decide how we use the above learnt info.

In [ ]:
df['savings_balance_gbp'].describe()

Investigating if there is any risk of data being skewed given the information we have to hand. (In this instance, luckily not drastically as the data shows only real zeroes) Noted the skew thanks to the high max-value.

In [ ]:
(df['savings_balance_gbp'] == 0).sum()

Re-confirmed the above.

In [ ]:
df.columns.tolist()

Assesed the column title againt the description file to check for other related columns. Also checked for proxy's and areas we should avoid relying on due to links to proxy's.

In [ ]:
df[['age_years',
    'years_at_current_address',]].corr()

### Linear Correlation

As you can see from the above we can see a correlation between age and years at current address but it is not a very strong one. Even so it needs further evidence to make a decision.

In [ ]:
df[['annual_income_gbp',
    'existing_debt_gbp',
    'loan_amount_gbp']].describe()

### Observations:

- We have genuine zeroes in existing debt which must be taken into account.
- Existing debt is skewed due to those genuine zeroes.
- There is a large range when looking at the requested loan amount.

In [ ]:
df['existing_debt_gbp'].div(
    df['annual_income_gbp']
).describe()

### Observations:

- Debt-to-income ratio is not a full picture of the applicants financial situation.
- We may also need to consider other expenses and how much, for how long they are requesting a loan for.
- However even adding monthly expenses could miss important factors if the applicant hasn't been fully honest.

### Priorities to take into next day:

- Reducing false negatives should be the focus for this project given the potential cost to the business.
- Pay close attention to recall for the default class.
- This may increase incorrect declines, this can have significant affects on applicants.
- This is not a final threshold decision just my thoughts.

In [ ]:
df.info()

In [ ]:
cols = [
    'annual_income_gbp',
    'monthly_expenses_gbp',
    'existing_debt_gbp',
    'loan_amount_gbp'
]

df[cols].describe()

In [ ]:
df['monthly_surplus'] = (
    df['annual_income_gbp'] / 12
    - df['monthly_expenses_gbp']
)

In [ ]:
df['monthly_surplus'].describe()

### Observations:

- Using these two variables we can interpret affordability in.
- We can see from the min value that an applicant has monthly expenses surpassing their monthly income.
- A lot of ouour applicants have a surplus of £200 or less, and this is not factoring in the proposed repayment amounts.
- This is only exploratory affordability.

In [ ]:
affordability_group = pd.cut(
    df['monthly_surplus'],
    bins=[float('-inf'), 0, 500, float('inf')],
    labels=['negative', '0-500', 'Above 500'],
    right=False,
)

In [ ]:
df['affordability_group'] = affordability_group

In [ ]:
df.groupby('affordability_group')['default_within_24m'].agg(['count', 'mean'])

### Observations:

- Grouped the affordability into some generic pools, this is just for exploration purposes.
- Then used these groups with the default result to determine default rate against affordability to assess the impact.
- The majority of defaults fit into the lower band.
- Whereas the majority of applicants fit into the upper band.
- These are not final groupings, just experimentation.

In [ ]:
missing_columns = [
    "savings_balance_gbp",
    "years_at_current_address"
]

print(df[missing_columns].isna().any(axis=1).sum())

In [ ]:
missing_mask = df[
    ["savings_balance_gbp", "years_at_current_address"]
].isna().any(axis=1)

df.groupby(missing_mask)["default_within_24m"].agg(
    ["count", "mean"]
)

### Notes:
The above two calculations are being used to determine if I should include the missing values in `savings_balance_gbp` and `years_at_current_address` in my model. As we can see from the results, 418 applicants in total have one or both of these factors missing from their application. When then looking at whether this could potentially impact on representation and bias we can see the difference in default rates is 1.75 percentage points. 

In [ ]:
df.groupby("num_existing_credit_lines")["default_within_24m"].agg(
    ["count", "mean"]
)

### Notes:
This was run to see if there was a correlation between more credit lines and more default risk. The simple answer is not really. However it is important to note that some groups have significantly less people in them therefore less reliable and that people with 8 credit lines have 0% default rates. Given the small group, this may be misleading.

In [ ]:
df.groupby("num_late_payments_24m")["default_within_24m"].agg(
    ["count", "mean"]
)

### Notes:
This is an experiment to see if we can see a correlation between late payments in the last 24 months and defaults and this is much clearer although we must be careful as after the 5th group you can see the number drops dramatically and there is a corresponding drop in mean too.

In [ ]:
df[df["months_in_current_job"] == 0]["employment_status"].value_counts()

In [ ]:
df[df["employment_status"] == "Unemployed"]["months_in_current_job"].value_counts()

Used the `value_counts` in both directions for `employment_status` and `months_in_current_job` to ensure the relationship between 0 and 'Unemployed' is linear and that the zeroes are structural not missing values.

In [ ]:
df["months_in_current_job"].describe()

Used `.describe()` to analyse where the boundaries are for `months_in_current_job` in order to choose boundaries for groups. Noted that the mean is higher than the median suggesting that the distribution may be is right-skewed. This also assisted with deciding that for experimenting with the data we should look at years for explainability when evaluating.

In [ ]:
employment_bins = [-1, 0, 11, 35, 59, float("inf")]

Creating bins to store boundaries for experimenting.

In [ ]:
employment_labels = [
    "No current tenure",
    "Under 1 year",
    "1-3 years",
    "3-5 years",
    "5+ years"
]

Creating labels to match the bins.

In [ ]:
employment_groups = pd.cut(
    df["months_in_current_job"],
    bins=employment_bins,
    labels=employment_labels
)

employment_groups.value_counts().sort_index()

The result of the last two cells. Further investigation to link to stakeholder needs will still be required.

In [ ]:
df.groupby(employment_groups, observed=True)["default_within_24m"].agg(
    ["count", "mean"]
)

A steady decrease in mean default rate observed alongside length of tenure. Must remember this is association not causation. This does not prove a linear relationship.